# Confound check

Images with a short side below `CROP_SIZE` (256) were bicubic-upscaled before the center crop (`data/preprocess.py`).

1. **How big is it?** Share of upscaled images per generator and label, read from `orig_w` / `orig_h` in the manifest.
2. **Can CLIP see it?** A linear probe on CLIP features of **real images only**, upscaled vs. not upscaled.
   AUROC ≈ 0.5 means the upscaling leaves no trace CLIP can use.
3. **Trace or content?** Control: take reals that were *not* upscaled, shrink and upscale them again, and see if the probe
   from 2 flags them. Same images before and after, so only the resampling differs.

In [1]:
import sys
sys.path.insert(0, "../..")  # repo root, so `data` is importable

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

from data import config
from data.features import load_features
from data.images import load_manifest

FEATURE_NAME = "clip_vit-l14"
TRAIN_POOL = "expert_train"  # probe is fitted on these reals
EVAL_POOL = "meta_val"  # and evaluated on these reals (test is not looked at)

## 1 · Share of upscaled images

In [2]:
manifest = load_manifest()
manifest["short_side"] = manifest[["orig_w", "orig_h"]].min(axis=1)
manifest["upscaled"] = manifest.short_side < config.CROP_SIZE
manifest["size"] = manifest.orig_w.astype(str) + "x" + manifest.orig_h.astype(str)
manifest["label_name"] = manifest.label.map(config.LABEL_NAMES)

reals = manifest[manifest.label == 0]
print(f"real images upscaled: {reals.upscaled.mean():.1%} ({reals.upscaled.sum()} of {len(reals)})")

manifest.groupby(["generator", "label_name"]).agg(
    upscaled_share=("upscaled", "mean"),
    median_short_side=("short_side", "median"),
    most_common_size=("size", lambda sizes: sizes.mode().iloc[0]),
    distinct_sizes=("size", "nunique"),
    images=("file", "size"),
).unstack("label_name").round(3)

real images upscaled: 8.7% (7627 of 88000)


upscaled_share        median_short_side         \
label_name                       fake   real              fake   real   
generator                                                               
ADM                               0.0  0.078             256.0  375.0   
BigGAN                            1.0  0.085             128.0  375.0   
Midjourney                        0.0  0.089            1024.0  375.0   
VQDM                              0.0  0.091             256.0  375.0   
glide                             0.0  0.092             256.0  375.0   
stable_diffusion_v_1_4            0.0  0.092             512.0  375.0   
stable_diffusion_v_1_5            0.0  0.074             512.0  375.0   
wukong                            0.0  0.092             512.0  375.0   

                       most_common_size          distinct_sizes       images  \
label_name                         fake     real           fake  real   fake   
generator                                                                      
ADM                             256x256  500x375              1  2488  11000   
BigGAN                          128x128  500x375              1  2564  11000   
Midjourney                    1024x1024  500x375              2  2621  11000   
VQDM                            256x256  500x375              1  2565  11000   
glide                           256x256  500x375              1  2684  11000   
stable_diffusion_v_1_4          512x512  500x375              1  2648  11000   
stable_diffusion_v_1_5          512x512  500x375              1  2434  11000   
wukong                          512x512  500x375              1  2651  11000   

                               
label_name               real  
generator                      
ADM                     11000  
BigGAN                  11000  
Midjourney              11000  
VQDM                    11000  
glide                   11000  
stable_diffusion_v_1_4  11000  
stable_diffusion_v_1_5  11000  
wukong                  11000

## 2 · Can CLIP see the upscaling? (real images only)

Same model as the CLIP experts: StandardScaler + LogisticRegression. The target is "was upscaled" and not real/fake,
so a high AUROC means the trace is in the features the experts use.

In [3]:
def upscaled_reals(pool):
    X, rows = load_features(FEATURE_NAME, pool=pool, label="real")
    return X, (rows[["orig_w", "orig_h"]].min(axis=1) < config.CROP_SIZE).to_numpy(), rows

X_train, y_train, rows_train = upscaled_reals(TRAIN_POOL)
X_eval, y_eval, rows_eval = upscaled_reals(EVAL_POOL)
print(f"train: {len(y_train)} reals, {y_train.mean():.1%} upscaled | eval: {len(y_eval)} reals, {y_eval.mean():.1%} upscaled")

probe = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000))
probe.fit(X_train, y_train)
p_upscaled = probe.predict_proba(X_eval)[:, 1]
print(f"AUROC upscaled vs. not upscaled ({EVAL_POOL}): {roc_auc_score(y_eval, p_upscaled):.4f}")

# Per generator: each generator has its own ImageNet reals.
for generator in config.GENERATORS:
    is_generator = (rows_eval.generator == generator).to_numpy()
    if len(set(y_eval[is_generator])) == 2:
        print(f"  {generator:24s} AUROC {roc_auc_score(y_eval[is_generator], p_upscaled[is_generator]):.4f}  "
              f"({y_eval[is_generator].mean():.1%} upscaled)")

train: 40000 reals, 10.0% upscaled | eval: 8000 reals, 9.9% upscaled
AUROC upscaled vs. not upscaled (meta_val): 0.9653
  ADM                      AUROC 0.9486  (8.5% upscaled)
  BigGAN                   AUROC 0.9629  (8.6% upscaled)
  glide                    AUROC 0.9825  (10.8% upscaled)
  Midjourney               AUROC 0.9579  (10.8% upscaled)
  stable_diffusion_v_1_4   AUROC 0.9788  (10.3% upscaled)
  stable_diffusion_v_1_5   AUROC 0.9555  (9.1% upscaled)
  VQDM                     AUROC 0.9766  (11.1% upscaled)
  wukong                   AUROC 0.9548  (9.9% upscaled)


## 3 · Control: resampling trace or content?

Small ImageNet photos may also *look* different (old, low-quality thumbnails), so the probe in 2 could partly detect content.
Here the content is fixed: reals of `EVAL_POOL` that were **not** upscaled are shrunk to a short side drawn from the
upscaled reals' sizes, bicubic-upscaled back to 256 (as in `data/preprocess.py`) and re-encoded as JPEG q95.
As baseline, the same images are only re-encoded as JPEG q95 (no resizing), so the extra JPEG pass is the same for both.

- **AUROC control vs. baseline high:** the probe sees the resampling itself, so the trace is real.
- **AUROC ≈ 0.5:** the signal in 2 came from content, not from the upscaling.

In [4]:
import io

import numpy as np
import torch
from PIL import Image

from clip.extract import load_model
from common.device import get_device
from data.images import open_image

N_CONTROL = 1000  # not-upscaled reals of EVAL_POOL to resample
BATCH_SIZE = 32

rng = np.random.default_rng(config.SEED)
not_upscaled = rows_eval[~y_eval].iloc[rng.permutation(int((~y_eval).sum()))[:N_CONTROL]]
# Short sides to shrink to: drawn from the real images that really were upscaled.
upscaled_short_sides = manifest[(manifest.label == 0) & manifest.upscaled].short_side.to_numpy()
short_sides = rng.choice(upscaled_short_sides, size=len(not_upscaled))
print(f"{len(not_upscaled)} images, short sides {short_sides.min()}-{short_sides.max()} (median {np.median(short_sides):.0f})")

W1103 15:33:26.449000 96327 site-packages/torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


1000 images, short sides 38-255 (median 181)


In [5]:
def jpeg(image):
    buffer = io.BytesIO()
    image.save(buffer, format="JPEG", quality=config.JPEG_QUALITY)
    return Image.open(buffer).convert("RGB")


def resample(image, short_side):
    """Shrink a 256x256 image to short_side, then bicubic-upscale back to 256 (our images are square)."""
    small = image.resize((short_side, short_side), Image.BICUBIC)
    return small.resize((config.CROP_SIZE, config.CROP_SIZE), Image.BICUBIC)


device = get_device()
model, preprocess = load_model(device)


def clip_features(images):
    features = []
    with torch.no_grad():
        for start in range(0, len(images), BATCH_SIZE):
            pixels = torch.stack([preprocess(image) for image in images[start:start + BATCH_SIZE]]).to(device)
            features.append(model.get_image_features(pixel_values=pixels).pooler_output.cpu().numpy())
    return np.concatenate(features)


originals = [open_image(file) for file in not_upscaled.file]
X_baseline = clip_features([jpeg(image) for image in originals])
X_control = clip_features([jpeg(resample(image, int(side))) for image, side in zip(originals, short_sides)])

p_baseline = probe.predict_proba(X_baseline)[:, 1]
p_control = probe.predict_proba(X_control)[:, 1]
labels = np.r_[np.zeros(len(p_baseline)), np.ones(len(p_control))]

print(f"mean p(upscaled): baseline {p_baseline.mean():.3f}, control {p_control.mean():.3f}")
print(f"flagged as upscaled (p > 0.5): baseline {(p_baseline > 0.5).mean():.1%}, control {(p_control > 0.5).mean():.1%}")
print(f"AUROC control vs. baseline: {roc_auc_score(labels, np.r_[p_baseline, p_control]):.4f}")
print(f"same image scored higher after resampling: {(p_control > p_baseline).mean():.1%}")

Loading weights:   0%|          | 0/590 [00:00<?, ?it/s]

mean p(upscaled): baseline 0.038, control 0.472
flagged as upscaled (p > 0.5): baseline 1.6%, control 45.6%
AUROC control vs. baseline: 0.9073
same image scored higher after resampling: 99.7%
